Проверим метрики для прогнозов gpt

В yandexgpt грузили обучающую выборку (с нарушением баланса классов и обогащением малых классов всеми доступными примерами - см. gpt_dataset.ipynb) на 50 тысяч примеров (train_v4.json) и использовали тест на 1721 пример с дисбалансом классов, из аналогичных тестов 4 и 6 собственной модели.  
В обучающие тексты добавили помимио назначений платежа - значения статьи, проекта и донора.

**Вывод**: На реальных данных LLM показал чудеса владения языком - очень хорошо выступила в редких классах, значительно превзойдя собственный catboost, но на мажоритарных классах сдала позиции, общая точность хуже.

In [1]:

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

y_true = pd.read_csv('y_test.csv', index_col=0)
y_pred_ygpt_ft = pd.read_csv('y_pred_ygpt_ft.csv',index_col=0)

In [2]:
# сверка индексов
print(y_true.index.equals(y_pred_ygpt_ft.index))

# проверка, что множества индексов совпадают (порядок не важен)
print(set(y_true.index) == set(y_pred_ygpt_ft.index))

# вывод различий
print("Есть в y_true, но нет в y_pred:", y_true.index.difference(y_pred_ygpt_ft.index))
print("Есть в y_pred, но нет в y_true:", y_pred_ygpt_ft.index.difference(y_true.index))

True
True
Есть в y_true, но нет в y_pred: Int64Index([], dtype='int64')
Есть в y_pred, но нет в y_true: Int64Index([], dtype='int64')


In [3]:
# считаем метрики для прогнозов YandexGPT дообученой на нашем датасете

accuracy = accuracy_score(y_true, y_pred_ygpt_ft)

precision = precision_score(y_true, y_pred_ygpt_ft, average='weighted')
recall = recall_score(y_true, y_pred_ygpt_ft, average='weighted')
f1 = f1_score(y_true, y_pred_ygpt_ft, average='weighted')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (weighted):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(y_true, y_pred_ygpt_ft))

Accuracy: 0.831493317838466
Precision (weighted): 0.8489719648333587
Recall (weighted): 0.831493317838466
F1-score (weighted): 0.8368497888964059

Отчет по классам:
                                             precision    recall  f1-score   support

                   гранты субсидии конкурсы       0.60      1.00      0.75         3
 пожертвования от физических лиц (напрямую)       0.91      0.91      0.91      1267
пожертвования от юридических лиц (напрямую)       0.36      0.62      0.45         8
              пожертвования через платформы       0.68      0.57      0.62       388
                            продажа товаров       0.16      1.00      0.28        10
                              продажа услуг       1.00      0.91      0.95        34
                 прочие недоходные операции       0.40      1.00      0.57         2
                          финансовые доходы       1.00      0.62      0.77         8
           членские и учредительские взносы       1.00      1.00     

In [4]:
y_true.value_counts()

universal_category                         
пожертвования от физических лиц (напрямую)     1267
пожертвования через платформы                   388
продажа услуг                                    34
продажа товаров                                  10
пожертвования от юридических лиц (напрямую)       8
финансовые доходы                                 8
гранты субсидии конкурсы                          3
прочие недоходные операции                        2
членские и учредительские взносы                  1
dtype: int64